# OmniParser v2 — Google Colab Demo

> **GPU 런타임 필수**: 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 저장

## 0. GPU 런타임 확인

In [9]:
import subprocess, sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    print('❌ GPU를 찾을 수 없습니다.')
    print('   상단 메뉴 → 런타임 → 런타임 유형 변경 → T4 GPU 선택 후 다시 실행하세요.')
    sys.exit(1)
print(result.stdout)

import torch
if not torch.cuda.is_available():
    print('❌ torch.cuda.is_available() = False')
    sys.exit(1)

print(f'✅ GPU 확인 완료')
print(f'   Device : {torch.cuda.get_device_name(0)}')
print(f'   VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print(f'   CUDA   : {torch.version.cuda}')

Sat May 30 17:03:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             30W /   70W |     401MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. 레포 클론 및 패키지 설치

In [10]:
import os, subprocess, shutil

repo_dir = '/content/OmniParser'

if os.path.exists(repo_dir):
    result = subprocess.run(
        ['git', '-C', repo_dir, 'remote', 'get-url', 'origin'],
        capture_output=True, text=True
    )
    if 'scythe0425' not in result.stdout:
        print('⚠️  이전 버전 감지 → 삭제 후 재클론')
        shutil.rmtree(repo_dir)
    else:
        !git -C {repo_dir} pull --quiet
        print('✅ 최신 코드로 업데이트 완료')

if not os.path.exists(repo_dir):
    !git clone https://github.com/scythe0425/OmniParser {repo_dir}
    print('✅ 클론 완료')

os.chdir(repo_dir)
print('작업 디렉토리:', os.getcwd())

✅ 최신 코드로 업데이트 완료
작업 디렉토리: /content/OmniParser


In [ ]:
# torch/torchvision/numpy: Colab 기본 탑재이므로 제외
!pip install -q \
    easyocr \
    transformers \
    ultralytics==8.3.70 \
    supervision==0.18.0 \
    timm \
    einops==0.8.0 \
    accelerate \
    paddlepaddle \
    paddleocr

## 2. 모델 가중치 다운로드

HuggingFace에서 OmniParser-v2.0 체크포인트를 다운로드합니다. (~1.5 GB)

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

os.makedirs('weights', exist_ok=True)

# OmniParser-v2.0 모델 가중치
model_files = [
    'icon_detect/train_args.yaml',
    'icon_detect/model.pt',
    'icon_detect/model.yaml',
    'icon_caption/config.json',
    'icon_caption/generation_config.json',
    'icon_caption/model.safetensors',
]
for f in model_files:
    hf_hub_download(repo_id='microsoft/OmniParser-v2.0', filename=f, local_dir='weights')
    print(f'✓ {f}')

if os.path.exists('weights/icon_caption') and not os.path.exists('weights/icon_caption_florence'):
    shutil.move('weights/icon_caption', 'weights/icon_caption_florence')

# Florence-2-base processor 파일 (tokenizer.json 등)
# weights/icon_caption_florence 에 함께 저장 → 로컬 로드 가능
proc_files = [
    'tokenizer.json', 'tokenizer_config.json',
    'special_tokens_map.json', 'preprocessor_config.json',
]
for f in proc_files:
    hf_hub_download(repo_id='microsoft/Florence-2-base', filename=f,
                    local_dir='weights/icon_caption_florence')
    print(f'✓ processor: {f}')

print('\n가중치 파일 확인:')
import subprocess
print(subprocess.run(
    ['find', 'weights', '-type', 'f', '-not', '-path', '*/.cache/*'],
    capture_output=True, text=True).stdout)

## 3. 모델 로드

In [13]:
import sys
sys.path.insert(0, '/content/OmniParser')

import torch
from PIL import Image
from util.utils import get_som_labeled_img, check_ocr_box, get_caption_model_processor, get_yolo_model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'사용 디바이스: {device}')

som_model = get_yolo_model('weights/icon_detect/model.pt')
som_model.to(device)
print('✅ YOLO 모델 로드 완료')

caption_model_processor = get_caption_model_processor(
    model_name='florence2',
    model_name_or_path='weights/icon_caption_florence',
    device=device
)
print('✅ Florence2 모델 로드 완료')

사용 디바이스: cuda
✅ YOLO 모델 로드 완료


You are using a model of type florence2 to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


AttributeError: TokenizersBackend has no attribute additional_special_tokens

## 4. 이미지 파싱

샘플 이미지로 테스트하거나, 직접 업로드한 스크린샷을 사용할 수 있습니다.

In [ ]:
import base64, io, time
import matplotlib.pyplot as plt

# 옵션 A: 샘플 이미지
image_path = 'imgs/windows_home.png'

# 옵션 B: 직접 업로드
# from google.colab import files
# uploaded = files.upload()
# image_path = list(uploaded.keys())[0]

image = Image.open(image_path)
print(f'이미지 크기: {image.size}')

box_overlay_ratio = max(image.size) / 3200
draw_bbox_config = {
    'text_scale': 0.8 * box_overlay_ratio,
    'text_thickness': max(int(2 * box_overlay_ratio), 1),
    'text_padding': max(int(3 * box_overlay_ratio), 1),
    'thickness': max(int(3 * box_overlay_ratio), 1),
}

t0 = time.time()
ocr_bbox_rslt, _ = check_ocr_box(
    image_path, display_img=False, output_bb_format='xyxy',
    goal_filtering=None,
    easyocr_args={'paragraph': False, 'text_threshold': 0.9},
    use_paddleocr=True
)
text, ocr_bbox = ocr_bbox_rslt
print(f'OCR 완료: {time.time()-t0:.1f}s')

t1 = time.time()
labeled_img_b64, label_coords, parsed_content_list = get_som_labeled_img(
    image_path, som_model,
    BOX_TRESHOLD=0.05,
    output_coord_in_ratio=True,
    ocr_bbox=ocr_bbox,
    draw_bbox_config=draw_bbox_config,
    caption_model_processor=caption_model_processor,
    ocr_text=text,
    use_local_semantics=True,
    iou_threshold=0.7,
    scale_img=False,
    batch_size=128
)
print(f'파싱 완료: {time.time()-t1:.1f}s  |  감지된 요소: {len(parsed_content_list)}개')

## 5. 결과 시각화

In [ ]:
import base64, io
import matplotlib.pyplot as plt
from PIL import Image

result_img = Image.open(io.BytesIO(base64.b64decode(labeled_img_b64)))
plt.figure(figsize=(16, 10))
plt.imshow(result_img)
plt.axis('off')
plt.title(f'OmniParser v2 결과 — {len(parsed_content_list)}개 요소 감지', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

df = pd.DataFrame(parsed_content_list)
df.index.name = 'ID'
df